### Limpieza del fichero 01_Ruido_diario_acumulado. Se carga desde Kaggle y se limpia y normalizan los datos

In [ ]:
!pip install reverse_geocoder

In [ ]:
import pandas as pd
import reverse_geocoder as rg
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.decomposition import PCA
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np



pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
file_path_01 = "01_Ruido_diario_acumulado.csv"

df_01 = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "raquelahdo/ruido-datasets",
  file_path_01,
    pandas_kwargs={"encoding": "latin-1", "sep": ";"}
)

In [ ]:
df_01.info()

In [ ]:
df_01.describe()

In [ ]:
missing_values_01 = df_01.isnull().sum()

display(missing_values_01)

In [ ]:
df_01.head()

Renombrado estandarizado de columnas: Nombres en minusculas, sin espacios, sin acentos, consistentes con Python.
Convertir tipos numéricos correctamente (a float)
Creación de columna Fecha.
Ordenar dataset por fecha.
Limpiar el campo "periodo" D (Diurno), E (Vespertino), N (Nocturno), T (Total)

In [ ]:
#creamos una columna fecha para trabajar
#renombramos las columnas con year, month y day. Y resto de columnas
df_01 = df_01.rename(columns={
    "Año": "year",
    "mes": "month",
    "dia": "day",
    "tipo": "periodo",
    "LAeq": "laeq",
    "L1": "l1",
    "L10": "l10",
    "L50": "l50",
    "L90": "l90",
    "L99": "l99"
})

#Se convierten las columnas numéricas a FLOAT (se elimina la coma por punto)
cols_numericas = ["laeq", "l1", "l10", "l50", "l90", "l99"]

for col in cols_numericas:
    df_01[col] = (
        df_01[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

#Se construye la columna Fecha (de tipo datetime64). Permite filtrado por año
#Agrupacion semanal/mensal y visualizaciones temporales
df_01["fecha"] = pd.to_datetime(
    df_01[["year", "month", "day"]],
    errors="coerce"
)

#Se ordena el dataset por fecha
df_01 = df_01.sort_values("fecha").reset_index(drop=True)

#Se limpia el campo periodo por valores de etiquetas más claros.
#Se crea columna periodo_desc con la dscripción del periodo
#D:Diurno, E: Vespertino, N: Nosturno, T: Total
df_01["periodo"] = df_01["periodo"].str.upper()
map_periodos = {
    "D": "Diurno",
    "E": "Vespertino",
    "N": "Nocturno",
    "T": "Total"
}
df_01["periodo_desc"] = df_01["periodo"].map(map_periodos)

#Se crean columnas derivadas útiles para gráficos
#Columna: mes en formato nombre
df_01["mes_nombre"] = df_01["fecha"].dt.month_name(locale="es_ES.utf8")

#Columna: Año-mes 
df_01["year_month"] = df_01["fecha"].dt.to_period("M").astype(str)

#Columna: semana del año
df_01["week"] = df_01["fecha"].dt.isocalendar().week

#Se eliminan filas con problemas en fecha y laeq (NaN). Deja dataset limpio y usable
df_01 = df_01.dropna(subset=["fecha", "laeq"])


In [ ]:
df_01.head()

In [ ]:
df_01.info()

NECESIDAD DE HABILITAR LA GESTIÓN MASIVA DE DATOS
The number of rows in your dataset is greater than the maximum allowed (5000).

Try enabling the VegaFusion data transformer which raises this limit by pre-evaluating data
transformations in Python.
    >> import altair as alt
    >> alt.data_transformers.enable("vegafusion")

In [ ]:
#Se activa VegaFusion
import altair as alt
alt.data_transformers.enable("vegafusion")


In [ ]:
pip install "vegafusion[embed]>=1.5.0"

In [ ]:
 pip install "vl-convert-python>=1.6.0"

In [ ]:

import altair as alt

# Desactivar motores que intentan usar vegafusion
alt.renderers.set_embed_options(actions=False)
alt.data_transformers.enable


In [ ]:
#VISUALIZAVIÓN 1: Tendencia temporal de LAeq (línea)

#Se reduce el tamañp deñ dataset para renderizarlo. 
#Se agregan los datos por día, lo que es viable para el análisis temporal.
#Reduce de 521K puntos a 12K puntos (12añosx365díasx3-4periodos)
df_daily = df_01.groupby(['fecha', 'periodo'], as_index=False)['laeq'].mean()


chart_tendencia = alt.Chart(df_daily).mark_line().encode(
    x=alt.X('fecha:T', title='Fecha'),
    y=alt.Y('laeq:Q', title='LAeq'),
    color=alt.Color('periodo:N', title='Periodo'),
    tooltip=['fecha:T', 'laeq:Q', 'periodo:N']
).properties(
    title='Evolución temporal del nivel acústico (LAeq) en Madrid',
    width=800,
    height=300
)

chart_tendencia

In [ ]:
#VISUALIZACIÓN 2: COMPARACIÓN ENTRE ESTACIONES - LAeq medio por estación (barras)

chart_estaciones = alt.Chart(df_01).mark_bar().encode(
    x='NMT:N',
    y='mean(laeq):Q',
    color='NMT:N',
    tooltip=['NMT:N', 'mean(laeq):Q']
).properties(
    title='Media de LAeq por estación'
)

chart_estaciones


In [ ]:
#VISUALIZACIÓN 3 - heatmap mensual (mes vs año)

#se crean columnas auxiliares
df_01["month_name"] = df_01["fecha"].dt.month_name(locale='es_ES.utf8')
df_01["year"] = df_01["fecha"].dt.year


chart_heatmap = alt.Chart(df_01).mark_rect().encode(
    x=alt.X('month(fecha):O', title='Mes'),
    y=alt.Y('year:O', title='Año'),
    color=alt.Color('mean(laeq):Q', title='LAeq medio', scale=alt.Scale(scheme='inferno')),
    tooltip=['year:O', 'month(fecha):O', 'mean(laeq):Q']
).properties(
    title='Mapa de calor del LAeq medio por mes y año',
    width=600,
    height=400
)

chart_heatmap


In [ ]:
#VISUALIZACIÓN 4: Bloxplots por estación

chart_box = alt.Chart(df_01).mark_boxplot().encode(
    x=alt.X('NMT:N', title='Estación'),
    y=alt.Y('laeq:Q', title='Distribución LAeq'),
    color='NMT:N',
    tooltip=['NMT:N', 'laeq:Q']
).properties(
    title='Distribución del nivel acústico (LAeq) por estación',
    width=700,
    height=400
)

chart_box


In [ ]:
#VISUALIZACIÓN 5: Selector y gráfica de tendencia


selector_estacion = alt.param(
    name='estacion',
    bind=alt.binding_select(options=df_01['NMT'].unique().tolist(), name='Estación:'),
    value=df_01['NMT'].iloc[0]   # valor inicial
)


chart_dashboard = alt.Chart(df_01).mark_line().encode(
    x='fecha:T',
    y='laeq:Q',
    color='NMT:N',
    tooltip=['fecha:T', 'laeq:Q', 'periodo:N']
).add_params(
    selector_estacion
).transform_filter(
    selector_estacion
).properties(
    title='Dashboard – Tendencia LAeq por estación seleccionada',
    width=800,
    height=300
)


chart_dashboard